# <font color='4361ee'> Notebook 9 - Final Model and Kaggle Predictions #

**Group 17**: Marta Silva (20241822), Beatriz Alves (20241747), Inês Claro (20241760), Pedro Ferreira (20241735)

In [183]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# scaling method
from sklearn.preprocessing import RobustScaler

# wrapper method
from xgboost import XGBClassifier

# imputation
from sklearn.impute import KNNImputer

# random over sampler
from imblearn.over_sampling import RandomOverSampler

## Preprocessing and Feature Selection pipeline

In this section we will follow the **approach** that resulted in the **best outcome** in the past notebooks. However, the results will **differ**, due to the way the train dataset is split from the start (in this notebook there will not be a subset for test, so as to have more **training** data).

**Step 1**: Import the datasets.

In [ ]:
# import datasets
learn = pd.read_csv("Nata_Files/learn.csv", index_col = 0) # training dataset
predict = pd.read_csv("Nata_Files/predict.csv", index_col = 0) # prediction dataset for kaggle submission

**Step 2**: Drop duplicate values and rows in the training set without target.

In [ ]:
learn.drop_duplicates(inplace=True)
learn = learn.dropna(axis = 0, subset = ['quality_class'])

**Step 3**: Divide the training dataset into the independent features and the target variable.

In [186]:
# split datasets into train (X) and target (y)
X = learn.drop('quality_class', axis=1)
target = learn['quality_class']

In this notebook the dataset is only split into `train` and `validation`, unlike the previous exploration notebooks where we had `train`, `validation`, and `test` sets, which means that from this point on the pipeline will not produce the same results as the Preprocessing/Feature Management/Modelling notebook, but rather uses the **parameters** and **methods** that produced the best results, to get equally high scores for this notebook's model.

**Step 4**: Split the training features and the target into train and validation subsets.

In [ ]:
# split X and y 
train, val, train_y, val_y = train_test_split(X, target, test_size=0.15, random_state=1, stratify = target)

**Step 5**: Use a Random Over Sampler to balance the dataset, by duplicating some of the rows from the minority class.

In [ ]:
# balance the dataset with oversampling
ros = RandomOverSampler(random_state=42)
train, train_y = ros.fit_resample(train, train_y)

**Step 6**: Add the features that are log transformations of the skewed original columns and remove the ones that do not provide any information.

In [189]:
# add new columns (log transformations for right-skewed features)
train['sugar_content_log'] = np.log(train['sugar_content'])
val['sugar_content_log'] = np.log(val['sugar_content'])
predict['sugar_content_log'] = np.log(predict['sugar_content'])

train['salt_ratio_log'] = np.log(train['salt_ratio'])
val['salt_ratio_log'] = np.log(val['salt_ratio'])
predict['salt_ratio_log'] = np.log(predict['salt_ratio'])

train['baking_duration_log'] = np.log(train['baking_duration'])
val['baking_duration_log'] = np.log(val['baking_duration'])
predict['baking_duration_log'] = np.log(predict['baking_duration'])

train['preheating_time_log'] = np.log(train['preheating_time'])
val['preheating_time_log'] = np.log(val['preheating_time'])
predict['preheating_time_log'] = np.log(predict['preheating_time'])

train['vanilla_extract_log'] = np.log(train['vanilla_extract'])
val['vanilla_extract_log'] = np.log(val['vanilla_extract'])
predict['vanilla_extract_log'] = np.log(predict['vanilla_extract'])

# drop the ones that aren't used to train the model
for col in ["notes_baker", "pastry_type"]:
    if col in train.columns:
        train = train.drop(columns=[col])
        print(f"Removed column from train: {col}")
    else:
        print(f"Column '{col}' already erased from train.")

for col in ["notes_baker", "pastry_type"]:
    if col in val.columns:
        val = val.drop(columns=[col])
        print(f"Removed column from val: {col}")
    else:
        print(f"Column '{col}' already erased from val.")

for col in ["notes_baker", "pastry_type"]:
    if col in predict.columns:
        predict = predict.drop(columns=[col])
        print(f"Removed column from predict: {col}")
    else:
        print(f"Column '{col}' already erased from predict.")


Removed column from train: notes_baker
Removed column from train: pastry_type
Removed column from val: notes_baker
Removed column from val: pastry_type
Removed column from predict: notes_baker
Removed column from predict: pastry_type


**Step 7**: Impute the missing values using K-Nearest Neighbors, starting by encoding the categorical column.

In [190]:
# uniformize categorical values and remove categorical NaNs
train_y = train_y[~train['origin'].isnull()]
train = train[~train['origin'].isnull()]
train['origin'] = train['origin'].str.strip().str.lower()
val['origin'] = val['origin'].str.strip().str.lower()
predict['origin'] = predict['origin'].str.strip().str.lower()

In [191]:
# encode the categorical column origin so it can be imputed
train['origin_encoded'] = train['origin'].map({'lisboa': 0, 'porto': 1})
val['origin_encoded'] = val['origin'].map({'lisboa': 0, 'porto': 1})
predict['origin_encoded'] = predict['origin'].map({'lisboa': 0, 'porto': 1})

train = train.drop(columns = "origin", axis = 1)
val = val.drop(columns = "origin", axis = 1)
predict = predict.drop(columns = "origin", axis = 1)

In [192]:
# impute the missing values using KNN
knn_imputer = KNNImputer(n_neighbors=6, weights='distance')
train_imputed = knn_imputer.fit_transform(train)
val_imputed = knn_imputer.transform(val)
predict_imputed = knn_imputer.transform(predict)

train = pd.DataFrame(train_imputed, columns = train.columns, index = train.index)
val = pd.DataFrame(val_imputed, columns= val.columns, index = val.index)
predict = pd.DataFrame(predict_imputed, columns= predict.columns, index = predict.index)

In [193]:
# decode origin after imputation
train['origin_encoded'] = (train['origin_encoded'] >= 0.5).astype(int)
val['origin_encoded'] =  (val['origin_encoded'] >= 0.5).astype(int)
predict['origin_encoded'] =  (predict['origin_encoded'] >= 0.5).astype(int)

train['origin'] = train['origin_encoded'].map({0: 'lisboa', 1: 'porto'})
val['origin'] = val['origin_encoded'].map({0: 'lisboa', 1: 'porto'})
predict['origin'] = predict['origin_encoded'].map({0: 'lisboa', 1: 'porto'})

train = train.drop(columns='origin_encoded', axis = 1)
val = val.drop(columns='origin_encoded', axis=1)
predict = predict.drop(columns='origin_encoded', axis=1)

**Step 8**: Change the datatypes to reduce memory usage and computational cost.

In [ ]:
# change datatypes

datatypes = {'ambient_humidity': 'int8', 'baking_duration': 'int8', 'cooling_period': 'int8', 'cream_fat_content': 'float32',
             'egg_temperature': 'int16', 'egg_yolk_count': 'int8', 'final_temperature': 'int16', 'lemon_zest_ph': 'float16', 
             'origin': 'category', 'oven_temperature': 'int16', 'preheating_time': 'int16', 'salt_ratio': 'float32', 
             'sugar_content': 'float32', 'vanilla_extract': 'float16', 'sugar_content_log': 'float16', 'salt_ratio_log': 'float16', 
             'baking_duration_log': 'float16', 'preheating_time_log': 'float16', 'vanilla_extract_log': 'float16'}

train = train.astype(datatypes)
val = val.astype(datatypes)
predict = predict.astype(datatypes)
train_y = train_y.astype('category')

**Step 9**: Scale the datasets using RobustScaler for better predictions in the final model.

In [195]:
# scale datasets
origin_t = train['origin']
origin_v = val['origin']
origin_p = predict['origin']
train = train.drop('origin', axis=1)
val = val.drop('origin', axis=1)
predict = predict.drop('origin', axis=1)

scaler = RobustScaler().fit(train)
# train data
train_scl = scaler.transform(train) 
train = pd.DataFrame(train_scl, columns = train.columns).set_index(train.index)

# validation data
val_scl = scaler.transform(val) 
val = pd.DataFrame(val_scl, columns = val.columns).set_index(val.index)

# prediction data
predict_scl = scaler.transform(predict) 
predict = pd.DataFrame(predict_scl, columns = predict.columns).set_index(predict.index)

train['origin'] = origin_t
val['origin'] = origin_v
predict['origin'] = origin_p

**Step 10**: Encode the columns that are not numerical so the model can interpret them.

In [196]:
# encode categorical features and target
train['origin'] = train['origin'].replace('porto', 0).replace('lisboa', 1)
val['origin'] = val['origin'].replace('porto', 0).replace('lisboa', 1)
predict['origin'] = predict['origin'].replace('porto', 0).replace('lisboa', 1)

train_y = train_y.replace('KO', 0).replace('OK', 1)
val_y = val_y.replace('KO', 0).replace('OK', 1)

## Modelling Pipeline

**Step 1**: Drop the features that were removed throughout the Feature Selection process.

In [ ]:
# remove the features that were not kept to train the final model
cols_drop = ['cream_fat_content', 'preheating_time', 'ambient_humidity', 'salt_ratio']

train = train.drop(columns=cols_drop, axis=1)
val = val.drop(columns=cols_drop, axis=1)
predict = predict.drop(columns=cols_drop, axis=1)

**Step 2**: Fit the chosen model (`XGBoostClassifier`) on the train set, with the validation as an evaluation during the learning.

In [ ]:
# fit the model
model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.009,
    max_depth=6,
    min_child_weight=5,
    subsample=0.80,
    colsample_bytree=0.7,
    colsample_bylevel=0.7,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method='hist',
    grow_policy='lossguide',
    enable_categorical=True, 
    early_stopping_rounds=150,
    eval_metric='error',
    n_jobs=-1,
    random_state=42
)

model.fit(train, train_y, 
        eval_set = [(val, val_y)],
        verbose = False)

print(model.get_params)

<bound method XGBModel.get_params of XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.7, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=150,
              enable_categorical=True, eval_metric='error', feature_types=None,
              feature_weights=None, gamma=0.1, grow_policy='lossguide',
              importance_type=None, interaction_constraints=None,
              learning_rate=0.009, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=-1, num_parallel_tree=None, ...)>


**Step 3**: Check the train set's predictions' score, which can indicate whether the model is overfitting or not when compared to the Kaggle score.

In [ ]:
# check train score
model.score(train, train_y)

0.8300738007380074

The **train score** was slightly higher on this notebook's model compared to our previous notebook's. However, the **public score**, which was our "test score" for this one, was also slightly higher than the last model's **test score**.

## Kaggle Predictions

**Step 1**: Use the fitted model to predict the target column of the `predict` dataset.

In [ ]:
# predict the target for kaggle
pd.Series(model.predict(predict), index=predict.index)

id
5201    0
5202    0
5203    0
5204    1
5205    1
       ..
6496    1
6497    1
6498    1
6499    1
6500    1
Length: 1300, dtype: int64

In [ ]:
# decode the predictions from 0s and 1s to KOs and OKs
encoded = pd.DataFrame(model.predict(predict), index=predict.index, columns=['quality_class'])
predictions = encoded.replace(0, 'KO').replace(1, 'OK')
predictions

,quality_class
id,
5201,KO
5202,KO
5203,KO
5204,OK
5205,OK
...,...
6496,OK
6497,OK
6498,OK


**Step 2**: Export the predictions to submit them on Kaggle.

In [202]:
# export kaggle predictions
predictions.to_csv('kaggle_predictions.csv')